# LABORATORIO N° 15 - Preparación del Proyecto Final
## FUNDAMENTOS DE GESTIÓN DE DATOS - Semana 15

**Docente:** Pilar Rocío Sayán Mejía  
**Semestre:** 2026-I

---

## ACTIVIDAD 1: Revisión de Conceptos - Proyecto Final

Complete la siguiente tabla reflexionando sobre el pipeline de datos desarrollado:

| Aspecto del Pipeline | Resumen de lo aplicado en tu proyecto |
|---------------------|---------------------------------------|
| 1. Fuente de datos y carga inicial | |
| 2. Modelo relacional y tablas en SQLite | |
| 3. Análisis exploratorio (EDA) realizado | |
| 4. Limpieza de datos aplicada | |
| 5. Modelo de ML implementado | |
| 6. Métricas de evaluación del modelo | |
| 7. Análisis de calidad de datos | |
| 8. Arquitectura propuesta | |
| 9. Principales hallazgos de negocio | |
| 10. Mejoras pendientes o trabajo futuro | |

---
## ACTIVIDAD 2: Revisión Integral del Pipeline

### Paso 1: Carga y verificación de la base de datos

In [ ]:
import pandas as pd
import sqlite3
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

print("Librerías cargadas correctamente")

In [ ]:
# Subir tu base de datos del proyecto
from google.colab import files
uploaded = files.upload()

# Conectar a la base de datos
db_name = list(uploaded.keys())[0]
conn = sqlite3.connect(db_name)

print(f"Conectado a: {db_name}")

In [ ]:
# Verificar estructura de la BD
print("="*60)
print("VERIFICACIÓN DE LA BASE DE DATOS")
print("="*60)

tablas = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f"\nTablas encontradas: {tablas['name'].tolist()}")

print("\nRegistros por tabla:")
total_registros = 0
for tabla in tablas['name']:
    count = pd.read_sql_query(f'SELECT COUNT(*) as n FROM {tabla}', conn)
    print(f"  - {tabla}: {count['n'][0]} registros")
    total_registros += count['n'][0]

print(f"\nTOTAL: {total_registros} registros en {len(tablas)} tablas")

In [ ]:
# Verificar esquema de cada tabla
print("\n" + "="*60)
print("ESQUEMA DE TABLAS")
print("="*60)

for tabla in tablas['name']:
    schema = pd.read_sql_query(f"PRAGMA table_info({tabla})", conn)
    print(f"\n{tabla}:")
    print(schema[['name', 'type', 'notnull', 'pk']].to_string(index=False))

**Pregunta 1:** ¿Tu base de datos tiene todas las tablas y registros esperados? ¿Hubo algún problema?

**Respuesta:**

### Paso 2: Revisión del modelo de ML

In [ ]:
# Cargar datos para el modelo
# EDITAR: Cambiar el nombre de la tabla y columnas según tu proyecto

tabla_modelo = 'nombre_tabla'  # <- EDITAR
df = pd.read_sql_query(f'SELECT * FROM {tabla_modelo}', conn)

print(f"Datos cargados: {df.shape}")
df.head()

In [ ]:
# Preparar datos para el modelo
# EDITAR: Seleccionar las columnas correctas

features = ['feature1', 'feature2', 'feature3']  # <- EDITAR
target = 'target'  # <- EDITAR

# Verificar si las columnas existen
print("Columnas disponibles:", df.columns.tolist())

# Filtrar nulos
df_model = df[features + [target]].dropna()
print(f"\nDatos para modelo: {len(df_model)} filas (sin nulos)")

In [ ]:
# Entrenar modelo
X = df_model[features]
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo = LinearRegression()
modelo.fit(X_train, y_train)

# Predicciones
y_pred = modelo.predict(X_test)

# Métricas
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print("="*60)
print("MÉTRICAS DEL MODELO")
print("="*60)
print(f"R² Score: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

# Interpretación
if r2 > 0.8:
    print("\n✓ Excelente: El modelo explica más del 80% de la varianza")
elif r2 > 0.6:
    print("\n⚠ Bueno: El modelo explica entre 60-80% de la varianza")
elif r2 > 0.4:
    print("\n⚠ Regular: El modelo necesita mejoras")
else:
    print("\n✗ Deficiente: El modelo no es útil para predicción")

In [ ]:
# Gráfico de predicción vs real
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Predicción vs Real
axes[0].scatter(y_test, y_pred, alpha=0.5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Valores Reales')
axes[0].set_ylabel('Predicciones')
axes[0].set_title(f'Predicción vs Real (R² = {r2:.4f})')

# Residuos
residuos = y_test - y_pred
axes[1].hist(residuos, bins=30, edgecolor='black')
axes[1].axvline(x=0, color='r', linestyle='--')
axes[1].set_xlabel('Residuos')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución de Residuos')

plt.tight_layout()
plt.savefig('modelo_evaluacion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Importancia de features (coeficientes)
coeficientes = pd.DataFrame({
    'Feature': features,
    'Coeficiente': modelo.coef_
}).sort_values('Coeficiente', key=abs, ascending=False)

print("\nIMPORTANCIA DE FEATURES:")
print(coeficientes.to_string(index=False))

**Pregunta 2:** ¿Cuál es el R² de tu modelo? ¿Es un valor aceptable? ¿Cómo podrías mejorarlo?

**Respuesta:**

### Paso 3: Resumen ejecutivo del proyecto

In [ ]:
# Generar resumen automático del proyecto
print("="*60)
print("RESUMEN EJECUTIVO DEL PROYECTO")
print("="*60)

# Editar con la información de tu proyecto
resumen = {
    'Problema de negocio': 'DESCRIBIR AQUÍ',
    'Fuente de datos': 'DESCRIBIR AQUÍ',
    'Volumen de datos': f'{total_registros} registros en {len(tablas)} tablas',
    'Modelo ML': 'Regresión Lineal',
    'R² del modelo': f'{r2:.4f}',
    'Principal hallazgo': 'DESCRIBIR AQUÍ',
    'Recomendación': 'DESCRIBIR AQUÍ'
}

for k, v in resumen.items():
    print(f"\n{k}:")
    print(f"  {v}")

---
## ACTIVIDAD 3: Preparación de la Exposición Final

### Estructura recomendada (10-15 minutos)

| Tiempo | Sección | Contenido |
|--------|---------|----------|
| 1-2 min | Introducción | Problema de negocio y equipo |
| 2-3 min | Datos | Fuentes, modelo ER, volumen |
| 3-4 min | Pipeline | EDA, limpieza, transformaciones |
| 3-4 min | Modelo ML | Algoritmo, métricas, resultados |
| 1-2 min | Demo en vivo | Consulta SQL o predicción |
| 1-2 min | Conclusiones | Hallazgos y recomendaciones |

### Consultas SQL para demostración en vivo

In [ ]:
# Consultas de demostración
# EDITAR según tu proyecto

print("CONSULTAS DE DEMOSTRACIÓN:")
print("="*60)

# Consulta 1: Resumen general
query1 = """
SELECT 'EJEMPLO' as descripcion, COUNT(*) as total
FROM nombre_tabla
GROUP BY alguna_columna
ORDER BY total DESC
LIMIT 5
"""

print("\nConsulta 1 (editar):")
print(query1)

# Ejecutar si está editada
# pd.read_sql_query(query1, conn)

In [ ]:
# Consulta 2: JOIN ejemplo
query2 = """
SELECT 
    t1.columna,
    COUNT(*) as cantidad,
    AVG(t2.valor) as promedio
FROM tabla1 t1
JOIN tabla2 t2 ON t1.id = t2.tabla1_id
GROUP BY t1.columna
ORDER BY cantidad DESC
"""

print("\nConsulta 2 (JOIN - editar):")
print(query2)

---
## PREGUNTAS FINALES

**Pregunta A:** ¿Cuál es el problema de negocio que aborda tu proyecto?

**Respuesta:**

**Pregunta B:** ¿Cuáles son los 3 hallazgos más importantes de tu análisis?

**Respuesta:**

1. 

2. 

3. 

**Pregunta C:** ¿Qué demostración en vivo realizarás? (consulta SQL, predicción del modelo, etc.)

**Respuesta:**

**Pregunta D:** ¿Qué aprendiste durante el curso que no sabías al inicio?

**Respuesta:**

**Pregunta E:** ¿Qué harías diferente si empezaras el proyecto de nuevo?

**Respuesta:**

---
## CONCLUSIONES

Escriba las conclusiones técnicas y de aprendizaje obtenidas tras la práctica:

1. 

2. 

3. 

In [ ]:
# Cerrar conexión
conn.close()
print("Conexión cerrada.")

In [ ]:
# Descargar archivos generados
from google.colab import files
files.download('modelo_evaluacion.png')